# World Model Training in Google Colab

This notebook trains the Phase 1 encoder-GRU-decoder world model using the existing repository code. It is designed for a fresh Google Colab runtime and keeps dataset generation, model definition, checkpointing, and evaluation aligned with the local project implementation.


## 1. Repository Setup

This section clones the repository, checks out the `dev` branch, moves into the project directory, and installs the project plus the notebook plotting dependency.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hilalaltunayy/partial-observation-world-model.git"
REPO_DIR = Path("/content/partial-observation-world-model")
BRANCH_NAME = "dev"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", BRANCH_NAME], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH_NAME], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "matplotlib"], check=True)

print(f"Working directory: {Path.cwd()}")


## 2. Runtime Detection

The notebook uses CUDA automatically when available and falls back to CPU otherwise.


In [ ]:
import platform

import torch

selected_device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {selected_device}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
else:
    print("GPU name: not available")


## 3. Dataset Preparation

The notebook checks for the generated train, validation, and test splits. If they are missing, it regenerates them with the existing deterministic dataset CLI using `300` episodes, `50` steps, and seed `42`.


In [ ]:
import json
import numpy as np

DATA_DIR = Path("data/generated")
split_paths = {
    "train": DATA_DIR / "train.npz",
    "validation": DATA_DIR / "validation.npz",
    "test": DATA_DIR / "test.npz",
}

missing_splits = [name for name, path in split_paths.items() if not path.exists()]
if missing_splits:
    print(f"Missing splits detected: {missing_splits}. Regenerating deterministic dataset...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "src.data.generate",
            "--episodes",
            "300",
            "--steps",
            "50",
            "--seed",
            "42",
        ],
        check=True,
    )
else:
    print("Using existing generated dataset splits.")

for split_name, split_path in split_paths.items():
    assert split_path.exists(), f"Expected dataset split file was not created: {split_path}"

for split_name, split_path in split_paths.items():
    print(f"\nSplit: {split_name}")
    with np.load(split_path, allow_pickle=False) as split_data:
        for array_name in split_data.files:
            array = split_data[array_name]
            print(f"  {array_name}: shape={array.shape}, dtype={array.dtype}")

manifest_path = DATA_DIR / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    print("\nManifest split sizes:")
    for split_name, transition_count in manifest["split_sizes"].items():
        print(f"  {split_name}: {transition_count} transitions")


## 4. Training Configuration

This is the single configuration cell to update when you want to change the Colab training run.


In [ ]:
seed = 42
epochs = 20
batch_size = 64
sequence_length = 8
learning_rate = 1e-3
latent_dimension = 64
gru_hidden_dimension = 96
loss_mode = "weighted_bce"
pos_weight_cap = 25.0
use_balanced_sampling = True
agent_sequence_sampling_weight = 4.0
device = selected_device
checkpoint_dir = Path("checkpoints/colab_world_model")

print("Training configuration:")
print(f"  seed: {seed}")
print(f"  epochs: {epochs}")
print(f"  batch size: {batch_size}")
print(f"  sequence length: {sequence_length}")
print(f"  learning rate: {learning_rate}")
print(f"  latent dimension: {latent_dimension}")
print(f"  GRU hidden dimension: {gru_hidden_dimension}")
print(f"  loss mode: {loss_mode}")
print(f"  pos-weight cap: {pos_weight_cap}")
print(f"  balanced sampling: {use_balanced_sampling}")
print(f"  agent-sequence sampling weight: {agent_sequence_sampling_weight}")
print(f"  device: {device}")
print(f"  checkpoint directory: {checkpoint_dir}")


## 5. Training Setup

The notebook reuses the repository's existing sequence dataset loader, world-model implementation, and training utilities. Assertions are included to fail early with clear messages if the dataset or tensor shapes are not what the Phase 1 pipeline expects.


In [ ]:
import matplotlib.pyplot as plt

from src.models import EncoderGruDecoderWorldModel, WorldModelConfig
from src.training import (
    TrainingConfig,
    build_dataloader,
    build_sequence_datasets,
    compute_channel_pos_weights,
    compute_training_split_channel_statistics,
    load_checkpoint,
    run_training,
    set_deterministic_seed,
)

set_deterministic_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

runtime_device = torch.device(device)
channel_statistics = compute_training_split_channel_statistics(DATA_DIR)
computed_pos_weights = compute_channel_pos_weights(channel_statistics, max_pos_weight_cap=pos_weight_cap)

print("Training split channel prevalence:")
for channel_name, stats in channel_statistics.channel_stats.items():
    print(
        f"  {channel_name}: positive_count={stats.positive_count}, "
        f"negative_count={stats.negative_count}, prevalence={stats.positive_prevalence:.6f}"
    )

print("Computed positive weights:")
for channel_name, weight in computed_pos_weights.items():
    print(f"  {channel_name}: {weight:.6f}")

datasets = build_sequence_datasets(data_dir=DATA_DIR, sequence_length=sequence_length, smoke_test=False)
train_dataset = datasets["train"]
validation_dataset = datasets["validation"]
test_dataset = datasets["test"]

assert len(train_dataset) > 0, "Training dataset produced no sequences. Check the generated data and sequence length."
assert len(validation_dataset) > 0, "Validation dataset produced no sequences. Check the generated data and sequence length."
assert len(test_dataset) > 0, "Test dataset produced no sequences. Check the generated data and sequence length."

sample = train_dataset[0]
assert sample["observations"].shape == (sequence_length, 4, 7, 7), (
    f"Unexpected observation sample shape: {tuple(sample['observations'].shape)}"
)
assert sample["actions"].shape == (sequence_length,), f"Unexpected action sample shape: {tuple(sample['actions'].shape)}"
assert sample["next_observations"].shape == (sequence_length, 4, 7, 7), (
    f"Unexpected next-observation sample shape: {tuple(sample['next_observations'].shape)}"
)

model_config = WorldModelConfig(
    observation_embedding_dim=latent_dimension,
    gru_hidden_dim=gru_hidden_dimension,
)
training_config = TrainingConfig(
    data_dir=str(DATA_DIR),
    epochs=epochs,
    batch_size=batch_size,
    sequence_length=sequence_length,
    learning_rate=learning_rate,
    device=device,
    seed=seed,
    checkpoint_dir=str(checkpoint_dir),
    smoke_test=False,
    loss_mode=loss_mode,
    pos_weight_cap=pos_weight_cap,
    use_balanced_sampling=use_balanced_sampling,
    agent_sequence_sampling_weight=agent_sequence_sampling_weight,
)

print("Sequence dataset summary:")
for split_name, dataset in datasets.items():
    metadata = getattr(dataset, "metadata", None)
    assert metadata is not None, f"Missing dataset metadata for split: {split_name}"
    print(
        f"  {split_name}: sequences={metadata.sequence_count}, episodes={metadata.episode_count}, "
        f"observation_shape={metadata.observation_shape}, action_count={metadata.action_count}, "
        f"agent_positive_sequences={metadata.scripted_agent_positive_sequence_count}"
    )


## 6. Train the World Model

This section performs epoch-by-epoch training, validates after every epoch, saves the best validation checkpoint, and records train and validation loss history for plotting.


In [ ]:
training_result = run_training(training_config, model_config=model_config)
history = training_result["history"]
best_checkpoint_path = training_result["best_checkpoint_path"]
best_metadata_path = training_result["best_metadata_path"]
datasets = training_result["datasets"]
validation_loader = build_dataloader(datasets["validation"], batch_size=batch_size, shuffle=False, seed=seed)
test_loader = build_dataloader(datasets["test"], batch_size=batch_size, shuffle=False, seed=seed)

assert best_checkpoint_path is not None and best_checkpoint_path.exists(), "Best checkpoint file was not created."
assert best_metadata_path is not None and best_metadata_path.exists(), "Checkpoint metadata file was not created."

best_epoch_record = max(history, key=lambda entry: (entry["validation_metrics"]["sparse_channel_mean_f1"], -entry["validation_metrics"]["loss"]))
print(f"\nBest checkpoint: {best_checkpoint_path}")
print(f"Best metadata: {best_metadata_path}")
print(
    f"Best validation objective (sparse_channel_mean_f1): "
    f"{best_epoch_record['validation_metrics']['sparse_channel_mean_f1']:.4f}"
)


## 7. Reload and Evaluate the Best Checkpoint

The best validation checkpoint is reloaded and evaluated on the validation and test splits. Metrics are reported for total loss, overall binary accuracy, and the four observation channels.


In [ ]:
channel_display_names = {
    "static_obstacles": "obstacles",
    "scripted_agents": "scripted agents",
    "observer_location": "observer",
    "out_of_bounds": "out-of-bounds / unknown",
}

reloaded_model = EncoderGruDecoderWorldModel(model_config).to(runtime_device)
checkpoint_payload = load_checkpoint(best_checkpoint_path, reloaded_model, optimizer=None, map_location=runtime_device)
validation_metrics = best_epoch_record["validation_metrics"]
test_metrics = training_result.get("test_metrics")
if test_metrics is None:
    from src.training import evaluate
    test_metrics = evaluate(
        reloaded_model,
        test_loader,
        runtime_device,
        loss_mode=loss_mode,
        pos_weight=None,
    )

def print_metric_block(split_name, metrics):
    print(f"\n{split_name} metrics")
    print(f"  loss: {metrics['loss']:.4f}")
    print(f"  overall binary accuracy: {metrics['overall_binary_accuracy']:.4f}")
    for channel_name in channel_display_names:
        print(
            f"  {channel_display_names[channel_name]}: "
            f"precision={metrics[f'{channel_name}_precision']:.4f} "
            f"recall={metrics[f'{channel_name}_recall']:.4f} "
            f"f1={metrics[f'{channel_name}_f1']:.4f} "
            f"iou={metrics[f'{channel_name}_iou']:.4f}"
        )

print(f"Checkpoint saved at epoch: {checkpoint_payload['epoch']}")
print_metric_block("Validation", validation_metrics)
print_metric_block("Test", test_metrics)


## 8. Plot Loss Curves

These curves give a compact view of training progress across epochs.


In [ ]:
train_loss_history = [entry["train_metrics"]["loss"] for entry in history]
validation_loss_history = [entry["validation_metrics"]["loss"] for entry in history]
epoch_numbers = [entry["epoch"] for entry in history]

assert len(train_loss_history) == epochs, "Training loss history length does not match the number of epochs."
assert len(validation_loss_history) == epochs, "Validation loss history length does not match the number of epochs."

plt.figure(figsize=(8, 4))
plt.plot(epoch_numbers, train_loss_history, marker="o", label="Train loss")
plt.plot(epoch_numbers, validation_loss_history, marker="o", label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("BCE with logits loss")
plt.title("World-model training and validation loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 9. Qualitative Prediction Comparison

The visual strips below compare the current observation, predicted next observation, real next observation, and absolute prediction error. Within each panel, channels are shown left to right in this order: obstacles, scripted agents, observer, out-of-bounds / unknown.


In [ ]:
def stack_observation_channels(observation_tensor):
    observation_array = np.asarray(observation_tensor)
    assert observation_array.shape == (4, 7, 7), f"Expected observation shape (4, 7, 7), got {observation_array.shape}"
    return np.concatenate([observation_array[channel_index] for channel_index in range(observation_array.shape[0])], axis=1)

reloaded_model.eval()
qualitative_batch = next(iter(validation_loader))
with torch.no_grad():
    qualitative_logits, _ = reloaded_model(
        qualitative_batch["observations"].to(runtime_device),
        qualitative_batch["actions"].to(runtime_device),
    )

predicted_next_observations = (torch.sigmoid(qualitative_logits) >= 0.5).float().cpu().numpy()
current_observations = qualitative_batch["observations"].cpu().numpy()
real_next_observations = qualitative_batch["next_observations"].cpu().numpy()

assert predicted_next_observations.shape == real_next_observations.shape, "Predictions and targets must have identical shapes."

examples_to_show = min(3, predicted_next_observations.shape[0])
time_index = sequence_length - 1
figure, axes = plt.subplots(examples_to_show, 4, figsize=(16, 3.5 * examples_to_show))
if examples_to_show == 1:
    axes = np.expand_dims(axes, axis=0)

panel_titles = [
    "Current observation",
    "Predicted next observation",
    "Real next observation",
    "Absolute prediction error",
]

for row_index in range(examples_to_show):
    current_observation = current_observations[row_index, time_index]
    predicted_observation = predicted_next_observations[row_index, time_index]
    real_observation = real_next_observations[row_index, time_index]
    absolute_error = np.abs(predicted_observation - real_observation)

    panel_images = [
        stack_observation_channels(current_observation),
        stack_observation_channels(predicted_observation),
        stack_observation_channels(real_observation),
        stack_observation_channels(absolute_error),
    ]

    for column_index, (panel_title, panel_image) in enumerate(zip(panel_titles, panel_images)):
        axis = axes[row_index, column_index]
        axis.imshow(panel_image, cmap="viridis", vmin=0.0, vmax=1.0)
        axis.set_title(panel_title)
        axis.set_xticks([])
        axis.set_yticks([])
        for boundary in (7, 14, 21):
            axis.axvline(boundary - 0.5, color="white", linewidth=0.75)
        if column_index == 0:
            axis.set_ylabel(f"Example {row_index + 1}")

figure.suptitle("Qualitative next-observation comparison", y=1.02)
figure.tight_layout()
plt.show()


## 10. Export the Best Checkpoint

These cells list the generated checkpoint files, bundle the best checkpoint and metadata into a zip archive, and trigger a Colab download.


In [ ]:
checkpoint_files = sorted(checkpoint_dir.glob("world_model_best*"))
assert checkpoint_files, f"No checkpoint files were found in {checkpoint_dir}"

print("Generated checkpoint files:")
for checkpoint_file in checkpoint_files:
    print(f"  {checkpoint_file} ({checkpoint_file.stat().st_size} bytes)")


In [ ]:
import zipfile

archive_path = checkpoint_dir / "world_model_best_bundle.zip"
assert best_checkpoint_path is not None and best_checkpoint_path.exists(), "Missing best checkpoint file for export."
assert best_metadata_path is not None and best_metadata_path.exists(), "Missing checkpoint metadata file for export."

with zipfile.ZipFile(archive_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    zip_file.write(best_checkpoint_path, arcname=best_checkpoint_path.name)
    zip_file.write(best_metadata_path, arcname=best_metadata_path.name)

print(f"Created export archive: {archive_path}")

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("google.colab is not available in this environment. Download the archive manually if needed.")


## Optional: Save to Google Drive

If you want to copy the exported checkpoint bundle into Drive, uncomment and run the example below.

```python
# from google.colab import drive
# drive.mount("/content/drive")
# drive_target = Path("/content/drive/MyDrive/partial-observation-world-model")
# drive_target.mkdir(parents=True, exist_ok=True)
# target_path = drive_target / archive_path.name
# target_path.write_bytes(archive_path.read_bytes())
# print(f"Copied archive to: {target_path}")
```
